<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pipeline_Custom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline_Compress
**Combined KD → Prune → KD-Repair → Quantise pipeline**

Applies all three compression techniques in a principled sequence on both
MobileNetV2 and MobileNetV3 best-seed checkpoints.

### Pipeline order (and why)

```
[seed checkpoint]
       │
  ① KD Warm-up   — raise the accuracy ceiling before destruction
       │
  ② Prune        — structured or unstructured, configurable sparsity
       │
  ③ KD Repair    — restore accuracy lost to pruning (lower LR!)
       │
  ④ Quantise     — INT8 QDQ ONNX + NSE gate
       │
  ⑤ STM32 block  — copy-paste stedgeai commands, memory budget check
```

**Why this order?**
- KD *before* pruning gives pruning a stronger starting point (already above the
  seed baseline), so the post-prune drop is shallower and recovery is faster.
- KD *after* pruning uses a much lower LR (`1e-4` for MV2, `3e-4` for MV3).
  Your logs proved that `1e-3` destroys MV2 checkpoints — the repair stage
  respects per-student LR limits.
- Quantisation runs last so the NSE score reflects the *fully compressed* model,
  not an intermediate state.

**Prerequisites:** run `Model_MobileNetV2`, `Model_MobileNetV3`, and
`Model_KD_Combined` first so the following checkpoints exist on Drive:
- `mobilenetv2_seed_74.pth`
- `mobilenetv3_seed_74.pth`
- `mv2_kd_vgg_ft.pth`  (used as warm-up start for MV2)
- `mv3_kd_vgg_ft.pth`  (used as warm-up start for MV3)

If the KD checkpoints are missing the pipeline falls back to the raw seed checkpoint.


In [1]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")


Mounted at /content/drive
✅ utils loaded from Drive


In [2]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

from pathlib import Path

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3, VGG_Pretrained
from utils.train   import setup_device, set_seed, evaluate, train_kd, train_epoch, validate_epoch

device = setup_device(seed=41)


Device: cuda


In [3]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")


1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation
✅ Dataset ready


In [4]:
# ── Config — edit here ──────────────────────────────────────────────
# ⚠️  Only this cell needs changing between runs.

SAVE_DIR   = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")
SHARED_DIR = EXPORT_DIR / "shared"

# ── Input checkpoints ──────────────────────────────────────────────
# Best seed checkpoints (fallback if KD checkpoints are missing)
SEED_CKPTS = {
    "MobileNetV2": f"{SAVE_DIR}/mobilenetv2_seed_74.pth",
    "MobileNetV3": f"{SAVE_DIR}/mobilenetv3_seed_74.pth",
}

# KD warm-up starting points (from Model_KD_Combined).
# If missing, pipeline falls back to SEED_CKPTS automatically.
KD_WARMUP_CKPTS = {
    "MobileNetV2": f"{SAVE_DIR}/mv2_kd_vgg_ft.pth",
    "MobileNetV3": f"{SAVE_DIR}/mv3_kd_vgg_ft.pth",
}

MODEL_FNS = {
    "MobileNetV2": VWW_MobileNetV2,
    "MobileNetV3": VWW_MobileNetV3,
}

# ── Which models to run ────────────────────────────────────────────
ACTIVE_MODELS = ["MobileNetV2", "MobileNetV3"]  # remove one to skip it

# ── Pruning config ─────────────────────────────────────────────────
PRUNE_TYPE   = "unstructured"   # "unstructured" | "structured"
PRUNE_AMOUNT = 0.30             # 0.10 / 0.20 / 0.30

# ── Stage ① KD Warm-up ─────────────────────────────────────────────
# Skipped automatically if KD_WARMUP_CKPTS[model] already exists and passes
# the accuracy threshold below.
# Set FORCE_KD_WARMUP = True to re-run even if checkpoint exists.
FORCE_KD_WARMUP   = False
KD_WARMUP_EPOCHS  = 60
KD_WARMUP_PATIENCE = 15
KD_WARMUP_LR      = {"MobileNetV2": 3e-4, "MobileNetV3": 1e-3}

# Accuracy threshold for skipping warmup (fraction, not percent)
KD_WARMUP_THRESHOLD = {"MobileNetV2": 0.795, "MobileNetV3": 0.810}

# ── Stage ③ KD Repair ──────────────────────────────────────────────
KD_REPAIR_EPOCHS   = 40
KD_REPAIR_PATIENCE = 12
# MV2: keep LR low — 1e-3 destroys checkpoints (confirmed in KD logs)
# MV3: needs slightly more to escape post-prune local minima
KD_REPAIR_LR = {"MobileNetV2": 1e-4, "MobileNetV3": 3e-4}

# ── Shared KD hyperparameters ──────────────────────────────────────
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_WD          = 1e-4

# ── Fine-tune (post-prune, pre-KD-repair) ─────────────────────────
FT_EPOCHS = 10
FT_LR     = 1e-4

# ── Quantisation ──────────────────────────────────────────────────
NSE_THRESHOLD = 0.95   # below this = do not deploy

print(f"Active models   : {ACTIVE_MODELS}")
print(f"Prune type      : {PRUNE_TYPE}  amount: {int(PRUNE_AMOUNT*100)}%")
print(f"KD Warm-up LR   : {KD_WARMUP_LR}")
print(f"KD Repair LR    : {KD_REPAIR_LR}")


Active models   : ['MobileNetV2', 'MobileNetV3']
Prune type      : unstructured  amount: 30%
KD Warm-up LR   : {'MobileNetV2': 0.0003, 'MobileNetV3': 0.001}
KD Repair LR    : {'MobileNetV2': 0.0001, 'MobileNetV3': 0.0003}


In [5]:
# ── Load Teacher (VGG — used in KD warm-up and repair) ──────────────
TEACHER_CKPT = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"

teacher = VGG_Pretrained().to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_acc = evaluate(teacher, val_loader, device)
print(f"Teacher VGG_Pretrained: {teacher_acc*100:.2f}%")


Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth


100%|██████████| 528M/528M [00:03<00:00, 151MB/s]


Teacher VGG_Pretrained: 89.07%


In [7]:
# ── Pruning helpers ──────────────────────────────────────────────────

def apply_unstructured(model, amount):
    """L1 unstructured: zeros individual weights in ALL Conv2d layers."""
    params = [(m, "weight") for m in model.modules() if isinstance(m, nn.Conv2d)]
    for layer, p in params:
        prune.l1_unstructured(layer, name=p, amount=amount)
    return params


def apply_structured(model, amount):
    """
    L2 structured (filter pruning): removes whole output filters (dim=0).
    Skips depthwise Conv2d (groups == in_channels) — breaking channel alignment
    inside inverted-residual blocks would corrupt the graph.
    """
    params = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and m.groups == 1:
            prune.ln_structured(m, name="weight", amount=amount, n=2, dim=0)
            params.append((m, "weight"))
    return params


def remove_pruning_masks(params):
    """Bake pruning masks into weights permanently before saving."""
    for layer, p in params:
        prune.remove(layer, p)


def compute_sparsity(model):
    """Fraction of zero weights across all Conv2d layers."""
    total = zeroed = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.detach().cpu().numpy()
            total  += w.size
            zeroed += (w == 0).sum()
    return zeroed / total if total > 0 else 0.0


# ── Standard fine-tune (used between prune and KD repair) ─────────────────────

def fine_tune(model, epochs, lr, label="FT"):
    set_seed(41)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=KD_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc   = 0.0
    best_state = None
    for epoch in range(1, epochs + 1):
        tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
        _,  va = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()
        marker = " ✅" if va > best_acc else ""
        print(f"  {label} {epoch:2d}/{epochs} | train {ta*100:.2f}% | val {va*100:.2f}%{marker}")
        if va > best_acc:
            best_acc   = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return best_acc


In [8]:
# ── Quantisation helpers (mirrors Pipeline_Quantz) ──────────────────
!pip -q install onnx onnxruntime onnxruntime-tools

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader


def skip_if_exists(path, label):
    if Path(path).exists():
        print(f"    ⏭️  {label} already exists")
        return True
    return False


def export_onnx(model, path):
    if skip_if_exists(path, "FP32 ONNX"): return
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"    ✅ FP32 ONNX: {path}")


def save_calib_npz(path, n=200):
    if skip_if_exists(path, "Calib data"): return
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
    np.savez(path, input=np.stack(xs))
    print(f"    ✅ Calib data: {path}")


def generate_shared_test_files(n=200):
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    inp_p = SHARED_DIR / "test_input.npz"
    lbl_p = SHARED_DIR / "test_labels.npz"
    if inp_p.exists() and lbl_p.exists():
        inp = np.load(inp_p)["input"]
        print(f"    ⏭️  Shared test files exist — {inp.shape}")
        return inp_p, lbl_p
    inputs, labels = [], []
    for i, (x, y) in enumerate(test_loader):
        if i >= n: break
        inputs.append(x.numpy().astype("float32")[0])
        labels.append(int(y.item()))
    np.savez(inp_p, input=np.stack(inputs))
    np.savez(lbl_p, label=np.array(labels, dtype="int32"))
    print(f"    ✅ Shared test files saved — {np.stack(inputs).shape}")
    return inp_p, lbl_p


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data): return None
        out = {"input": self.data[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0


def quantize_int8(fp32_path, calib_path, int8_path):
    if skip_if_exists(int8_path, "INT8 ONNX"): return
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"    ✅ INT8 QDQ ONNX: {int8_path}")


def compute_nse(fp32_path, int8_path, input_npz):
    inputs = np.load(input_npz)["input"]
    s32  = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    s8   = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    fp32_outs, int8_outs = [], []
    for i in range(len(inputs)):
        sample = inputs[i:i+1]
        fp32_outs.append(s32.run(["logits"], {"input": sample})[0][0])
        int8_outs.append(s8.run(["logits"],  {"input": sample})[0][0])
    fp32_outs = np.array(fp32_outs)
    int8_outs = np.array(int8_outs)
    num = np.sum((fp32_outs - int8_outs) ** 2)
    den = np.sum((fp32_outs - fp32_outs.mean()) ** 2)
    return float(1 - num / den)


print("✅ Quantisation helpers loaded")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.5 MB/s eta 0:00:00
✅ Quantisation helpers loaded


In [9]:
# ── Pre-flight checkpoint check ─────────────────────────────────────
print(f"{'Checkpoint':<55} {'Status'}")
print("-" * 70)

for model_name in ACTIVE_MODELS:
    seed_path   = SEED_CKPTS[model_name]
    warmup_path = KD_WARMUP_CKPTS[model_name]
    seed_ok     = os.path.exists(seed_path)
    warmup_ok   = os.path.exists(warmup_path)
    print(f"  {'✅' if seed_ok   else '❌'}  {model_name} seed     {seed_path}")
    print(f"  {'✅' if warmup_ok else '⚠️ '} {model_name} kd_warmup {warmup_path}")
    if not seed_ok:
        raise FileNotFoundError(f"STOP — seed checkpoint missing: {seed_path}")
    if not warmup_ok:
        print(f"       └─ Will fall back to seed checkpoint for {model_name}")

print(f"\n  {'✅' if os.path.exists(TEACHER_CKPT) else '❌'}  Teacher  {TEACHER_CKPT}")
if not os.path.exists(TEACHER_CKPT):
    raise FileNotFoundError(f"STOP — teacher checkpoint missing: {TEACHER_CKPT}")

print("\n✅ Pre-flight passed")


Checkpoint                                              Status
----------------------------------------------------------------------
  ✅  MobileNetV2 seed     /content/drive/My Drive/stm32-thesis/checkpoints/mobilenetv2_seed_74.pth
  ✅ MobileNetV2 kd_warmup /content/drive/My Drive/stm32-thesis/checkpoints/mv2_kd_vgg_ft.pth
  ✅  MobileNetV3 seed     /content/drive/My Drive/stm32-thesis/checkpoints/mobilenetv3_seed_74.pth
  ✅ MobileNetV3 kd_warmup /content/drive/My Drive/stm32-thesis/checkpoints/mv3_kd_vgg_ft.pth

  ✅  Teacher  /content/drive/My Drive/stm32-thesis/checkpoints/vgg_pretrained_seed_85.pth

✅ Pre-flight passed


In [10]:
# ── Main pipeline ────────────────────────────────────────────────────
# Runs each active model through all 5 stages.
# Results are collected in `pipeline_records` for the summary table.

SHARED_INPUT_NPZ, SHARED_LABELS_NPZ = generate_shared_test_files()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_records = []

for model_name in ACTIVE_MODELS:
    model_fn   = MODEL_FNS[model_name]
    seed_ckpt  = SEED_CKPTS[model_name]
    prune_tag  = f"{PRUNE_TYPE}_{int(PRUNE_AMOUNT*100)}pct"

    print(f"\n{'═'*66}")
    print(f"  {model_name}  |  {prune_tag}")
    print(f"{'═'*66}")

    # Output checkpoint paths
    out_warmup  = f"{SAVE_DIR}/{model_name.lower()}_compress_warmup.pth"
    out_pruned  = f"{SAVE_DIR}/{model_name.lower()}_compress_pruned_{prune_tag}.pth"
    out_repaired = f"{SAVE_DIR}/{model_name.lower()}_compress_final_{prune_tag}.pth"
    out_dir     = EXPORT_DIR / f"{model_name.lower()}_compress_{prune_tag}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ──────────────────────────────────────────────────────────────────
    # ① KD WARM-UP
    # ──────────────────────────────────────────────────────────────────
    print("\n▶ Stage ①  KD Warm-up")

    # Decide starting point: prefer existing KD warmup, else seed checkpoint
    warmup_source = KD_WARMUP_CKPTS[model_name]
    if not os.path.exists(warmup_source):
        warmup_source = seed_ckpt
        print(f"  KD warmup checkpoint missing — using seed: {warmup_source}")

    # Load model, measure baseline
    _tmp = model_fn().to(device)
    _tmp.load_state_dict(torch.load(warmup_source, map_location=device))
    warmup_baseline = evaluate(_tmp, val_loader, device)
    del _tmp
    print(f"  Starting accuracy: {warmup_baseline*100:.2f}%")

    # Check if we should skip warmup training
    run_warmup = True
    if not FORCE_KD_WARMUP and os.path.exists(out_warmup):
        _tmp = model_fn().to(device)
        _tmp.load_state_dict(torch.load(out_warmup, map_location=device))
        existing_acc = evaluate(_tmp, val_loader, device)
        del _tmp
        threshold = KD_WARMUP_THRESHOLD[model_name]
        if existing_acc >= threshold:
            print(f"  ⏭️  Skipping — warmup checkpoint exists: {existing_acc*100:.2f}% >= {threshold*100:.1f}%")
            warmup_acc = existing_acc
            run_warmup = False
        else:
            print(f"  Checkpoint below threshold ({existing_acc*100:.2f}% < {threshold*100:.1f}%) — re-training")

    if run_warmup:
        student = model_fn().to(device)
        student.load_state_dict(torch.load(warmup_source, map_location=device))
        warmup_acc, _ = train_kd(
            student      = student,
            teacher      = teacher,
            train_loader = train_loader,
            val_loader   = val_loader,
            device       = device,
            epochs       = KD_WARMUP_EPOCHS,
            lr           = KD_WARMUP_LR[model_name],
            weight_decay = KD_WD,
            temperature  = KD_TEMPERATURE,
            alpha        = KD_ALPHA,
            patience     = KD_WARMUP_PATIENCE,
            save_path    = out_warmup,
        )
    print(f"  ✅ Warm-up acc: {warmup_acc*100:.2f}%  (delta from start: {(warmup_acc - warmup_baseline)*100:+.2f}%)")

    # ──────────────────────────────────────────────────────────────────
    # ② PRUNE
    # ──────────────────────────────────────────────────────────────────
    print(f"\n▶ Stage ②  {PRUNE_TYPE.title()} Pruning @ {int(PRUNE_AMOUNT*100)}%")

    model = model_fn().to(device)
    model.load_state_dict(torch.load(out_warmup, map_location=device))
    pre_prune_acc = evaluate(model, val_loader, device)
    print(f"  Pre-prune accuracy: {pre_prune_acc*100:.2f}%")

    if PRUNE_TYPE == "unstructured":
        applied = apply_unstructured(model, PRUNE_AMOUNT)
    else:
        applied = apply_structured(model, PRUNE_AMOUNT)

    post_prune_acc  = evaluate(model, val_loader, device)
    actual_sparsity = compute_sparsity(model)
    print(f"  After pruning (no FT): {post_prune_acc*100:.2f}%  "
          f"sparsity: {actual_sparsity*100:.1f}%  "
          f"drop: {(post_prune_acc - pre_prune_acc)*100:+.2f}%")

    # Quick standard fine-tune to partially recover before KD repair
    print(f"  Running {FT_EPOCHS}-epoch fine-tune before KD repair...")
    ft_acc = fine_tune(model, FT_EPOCHS, FT_LR, label="FT")
    print(f"  Post-FT acc: {ft_acc*100:.2f}%")

    # Bake masks and save pruned checkpoint
    remove_pruning_masks(applied)
    torch.save(model.state_dict(), out_pruned)
    print(f"  ✅ Pruned checkpoint saved → {out_pruned}")

    # ──────────────────────────────────────────────────────────────────
    # ③ KD REPAIR
    # ──────────────────────────────────────────────────────────────────
    print(f"\n▶ Stage ③  KD Repair (LR={KD_REPAIR_LR[model_name]})")

    repaired_acc, _ = train_kd(
        student      = model,          # continue from fine-tuned pruned model
        teacher      = teacher,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_REPAIR_EPOCHS,
        lr           = KD_REPAIR_LR[model_name],
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_REPAIR_PATIENCE,
        save_path    = out_repaired,
    )
    print(f"  ✅ Repaired acc: {repaired_acc*100:.2f}%  "
          f"(vs warm-up: {(repaired_acc - warmup_acc)*100:+.2f}%  "
          f"vs pre-prune: {(repaired_acc - pre_prune_acc)*100:+.2f}%)")

    # Reload best repaired state
    model.load_state_dict(torch.load(out_repaired, map_location=device))
    model.eval()

    # ──────────────────────────────────────────────────────────────────
    # ④ QUANTISE (FP32 ONNX → INT8 QDQ ONNX → NSE check)
    # ──────────────────────────────────────────────────────────────────
    print("\n▶ Stage ④  Quantisation")

    fp32_path  = out_dir / "model_fp32.onnx"
    calib_path = out_dir / "calib_train.npz"
    int8_path  = out_dir / "model_int8_qdq.onnx"

    export_onnx(model, fp32_path)
    save_calib_npz(calib_path)
    quantize_int8(fp32_path, calib_path, int8_path)

    nse = compute_nse(fp32_path, int8_path, SHARED_INPUT_NPZ)
    nse_ok = nse >= NSE_THRESHOLD
    nse_verdict = "✅ DEPLOY" if nse_ok else "⛔ DO NOT DEPLOY"
    print(f"  NSE: {nse:.4f}  →  {nse_verdict}")

    if not nse_ok:
        print(f"  ⚠️  NSE < {NSE_THRESHOLD} — quantisation quality is poor.")
        print("     Options: (a) increase KD repair epochs, (b) reduce prune amount,")
        print("              (c) switch prune type (try structured if running unstructured)")

    # Collect results
    pipeline_records.append({
        "model"           : model_name,
        "prune_type"      : PRUNE_TYPE,
        "prune_%"         : int(PRUNE_AMOUNT * 100),
        "sparsity_%"      : round(actual_sparsity * 100, 1),
        "warmup_%"        : round(warmup_acc * 100, 2),
        "pre_prune_%"     : round(pre_prune_acc * 100, 2),
        "post_prune_%"    : round(post_prune_acc * 100, 2),
        "post_FT_%"       : round(ft_acc * 100, 2),
        "repaired_%"      : round(repaired_acc * 100, 2),
        "NSE"             : round(nse, 4),
        "deploy"          : nse_ok,
        "fp32_path"       : str(fp32_path),
        "int8_path"       : str(int8_path),
        "out_dir"         : str(out_dir),
    })

    del model
    print(f"\n  {'='*60}")
    print(f"  {model_name} pipeline complete.")
    print(f"  {'='*60}")

print("\n\n✅ All models processed.")


    ⏭️  Shared test files exist — (200, 3, 96, 96)

══════════════════════════════════════════════════════════════════
  MobileNetV2  |  unstructured_30pct
══════════════════════════════════════════════════════════════════

▶ Stage ①  KD Warm-up
  Starting accuracy: 80.00%
Initial student accuracy: 80.00%
Epoch   1/60 | Train 82.81% | Val 77.40%
Epoch   2/60 | Train 82.97% | Val 77.27%
Epoch   3/60 | Train 83.16% | Val 78.87%
Epoch   4/60 | Train 82.66% | Val 79.47%
Epoch   5/60 | Train 82.93% | Val 78.53%
Epoch   6/60 | Train 83.33% | Val 78.27%
Epoch   7/60 | Train 83.11% | Val 78.33%
Epoch   8/60 | Train 84.23% | Val 77.40%
Epoch   9/60 | Train 83.20% | Val 78.80%
Epoch  10/60 | Train 83.86% | Val 78.60%
Epoch  11/60 | Train 83.29% | Val 78.67%
Epoch  12/60 | Train 84.30% | Val 78.87%
Epoch  13/60 | Train 83.59% | Val 78.13%
Epoch  14/60 | Train 84.67% | Val 79.13%
Epoch  15/60 | Train 83.93% | Val 78.60%
🛑 Early stopping at epoch 15

✅ Best KD val acc: 80.00%  (3.9 min)
  ✅ Warm-up

/tmp/ipykernel_1027/673768091.py:20: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


    ✅ FP32 ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_compress_unstructured_30pct/model_fp32.onnx


    ✅ Calib data: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_compress_unstructured_30pct/calib_train.npz


    ✅ INT8 QDQ ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_compress_unstructured_30pct/model_int8_qdq.onnx
  NSE: 0.9807  →  ✅ DEPLOY

  MobileNetV2 pipeline complete.

══════════════════════════════════════════════════════════════════
  MobileNetV3  |  unstructured_30pct
══════════════════════════════════════════════════════════════════

▶ Stage ①  KD Warm-up
  Starting accuracy: 79.13%
Initial student accuracy: 79.13%
Epoch   1/60 | Train 79.70% | Val 76.53%
Epoch   2/60 | Train 79.34% | Val 76.67%
Epoch   3/60 | Train 80.16% | Val 74.60%
Epoch   4/60 | Train 79.54% | Val 77.07%
Epoch   5/60 | Train 80.73% | Val 74.73%
Epoch   6/60 | Train 80.10% | Val 77.67%
Epoch   7/60 | Train 80.50% | Val 75.00%
Epoch   8/60 | Train 79.79% | Val 77.53%
Epoch   9/60 | Train 80.63% | Val 77.80%
Epoch  10/60 | Train 80.96% | Val 76.93%
Epoch  11/60 | Train 80.36% | Val 78.00%
Epoch  12/60 | Train 81.36% | Val 76.93%
Epoch  13/60 | Train 81.40% | Val 77.93%
Epoch  14/60 | Train 81.

    ✅ Calib data: /content/drive/My Drive/stm32-thesis/exports/mobilenetv3_compress_unstructured_30pct/calib_train.npz


    ✅ INT8 QDQ ONNX: /content/drive/My Drive/stm32-thesis/exports/mobilenetv3_compress_unstructured_30pct/model_int8_qdq.onnx
  NSE: 0.9884  →  ✅ DEPLOY

  MobileNetV3 pipeline complete.


✅ All models processed.


In [11]:
# ── Results summary ─────────────────────────────────────────────────
df = pd.DataFrame(pipeline_records)

display_cols = [
    "model", "prune_type", "prune_%", "sparsity_%",
    "warmup_%", "pre_prune_%", "post_prune_%", "post_FT_%",
    "repaired_%", "NSE", "deploy",
]
print("\n" + "="*90)
print("COMPRESSION PIPELINE RESULTS")
print("="*90)
print(df[display_cols].to_string(index=False))
print("="*90)

# Best deployable model
deployable = df[df["deploy"] == True]
if len(deployable) > 0:
    best = deployable.loc[deployable["repaired_%"].idxmax()]
    print(f"\n🏆 Best deployable: {best['model']} | {best['prune_type']} {int(best['prune_%'])}%")
    print(f"   Repaired acc: {best['repaired_%']}%   NSE: {best['NSE']}")
    print(f"   INT8 ONNX  : {best['int8_path']}")
else:
    print("\n⚠️  No model passed the NSE gate.")
    print("   Try: reduce PRUNE_AMOUNT, increase KD_REPAIR_EPOCHS, or switch PRUNE_TYPE.")



COMPRESSION PIPELINE RESULTS
      model   prune_type  prune_%  sparsity_%  warmup_%  pre_prune_%  post_prune_%  post_FT_%  repaired_%    NSE  deploy
MobileNetV2 unstructured       30        30.0     80.00        80.00         65.33      79.07       80.47 0.9807    True
MobileNetV3 unstructured       30        30.0     79.13        79.13         65.87      78.47       78.47 0.9884    True

🏆 Best deployable: MobileNetV2 | unstructured 30%
   Repaired acc: 80.47%   NSE: 0.9807
   INT8 ONNX  : /content/drive/My Drive/stm32-thesis/exports/mobilenetv2_compress_unstructured_30pct/model_int8_qdq.onnx


## ⑤ STM32N6 Deployment Block

Run these commands **locally on Windows** after downloading the INT8 ONNX files.
Update the path to the `int8_path` you got from the results table above.

```
stedgeai.exe generate \
  --target stm32n6 \
  --name network \
  -m <int8_path> \
  --st-neural-art n6-allmems-O3@<X-CUBE-AI>/scripts/N6_scripts/user_neuralart.json \
  --workspace <tmp_dir> \
  --output <network_output>
```

```
stedgeai.exe validate \
  --target stm32n6 \
  --name network \
  -m <int8_path> \
  --valset-path <shared_test_dir>/test_input.npz \
  --output <network_output>
```

### Memory budget reference (STM32N6)

| Region         | Available   | Typical usage (compressed MV2) |
|----------------|-------------|--------------------------------|
| octoFlash (RO) | 112 MB      | ~170–234 KiB (weights)          |
| npuRAM5 (RW)   | 448 KiB     | ~270 KiB (activations)          |
| Total          | —           | < 504 KiB comfortably fits      |

NSE ≥ 0.95 is required before running `validate` — a low NSE means
the INT8 inference will be unreliable and the on-device accuracy will
not match the Colab evaluation.


In [12]:
# ── Debug: layer-by-layer accuracy sensitivity after pruning ────────
# Run this cell if the pipeline produced a poor NSE or large accuracy drop.
# It identifies which layers lose the most accuracy when pruned in isolation.
# Use this to decide which layers to EXCLUDE from pruning.

DEBUG_MODEL = "MobileNetV2"   # change to "MobileNetV3" if needed

_fn   = MODEL_FNS[DEBUG_MODEL]
_ckpt = pipeline_records[[r["model"] for r in pipeline_records].index(DEBUG_MODEL)]["fp32_path"]

# We need the .pth, not the ONNX. Use out_warmup path.
_warmup_pth = f"{SAVE_DIR}/{DEBUG_MODEL.lower()}_compress_warmup.pth"

_model = _fn().to(device)
_model.load_state_dict(torch.load(_warmup_pth, map_location=device))
_base  = evaluate(_model, val_loader, device)
print(f"{DEBUG_MODEL} baseline: {_base*100:.2f}%\n")

print(f"{'Layer':<60} {'Post-prune acc':>15}  {'Drop':>8}")
print("-" * 90)

conv_layers = [(name, m) for name, m in _model.named_modules() if isinstance(m, nn.Conv2d)]

for layer_name, layer_mod in conv_layers:
    # Temporarily prune just this layer
    _h = prune.l1_unstructured(layer_mod, name="weight", amount=PRUNE_AMOUNT)
    _acc = evaluate(_model, val_loader, device)
    _drop = (_acc - _base) * 100
    flag = " ⚠️" if _drop < -2.0 else ""
    print(f"  {layer_name:<58} {_acc*100:>14.2f}%  {_drop:>+7.2f}%{flag}")
    prune.remove(layer_mod, "weight")  # restore

del _model
print("\n✅ Sensitivity analysis complete. High-drop layers should be excluded from pruning.")


MobileNetV2 baseline: 80.00%

Layer                                                         Post-prune acc      Drop
------------------------------------------------------------------------------------------
  initial.0                                                           77.73%    -2.27% ⚠️
  features.0.block.0                                                  77.33%    -2.67% ⚠️
  features.0.block.3                                                  75.53%    -4.47% ⚠️
  features.0.block.6                                                  73.93%    -6.07% ⚠️
  features.1.block.0                                                  74.07%    -5.93% ⚠️
  features.1.block.3                                                  72.93%    -7.07% ⚠️
  features.1.block.6                                                  72.53%    -7.47% ⚠️
  features.2.block.0                                                  70.33%    -9.67% ⚠️
  features.2.block.3                                                  70